# 🤖 Telegram бот — Сидорович из S.T.A.L.K.E.R.
Запускай ячейки по порядку сверху вниз.

## ⚙️ Шаг 1 — Установка зависимостей
Запусти один раз, потом можно пропускать.

In [1]:
!pip install python-telegram-bot==21.6 httpx -q


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 🔧 Шаг 2 — Настройки
Вставь свой токен и название модели из LM Studio.

In [2]:
TELEGRAM_TOKEN = "8697925205:AAET64g_C3u1-Q53omGBcZfFms3RbHl3qQ0"       # токен от @BotFather
LM_STUDIO_URL  = "http://localhost:1234/v1/chat/completions"
MODEL_NAME     = "qwen/qwen3.5-9b"       # точное имя из LM Studio
MAX_HISTORY    = 20                       # сколько сообщений помнить
TEMPERATURE    = 0.7
MAX_TOKENS     = 4096                     # УВЕЛИЧЕНО - для thinking mode нужно больше
TG_LIMIT       = 4000                     # лимит символов в одном сообщении Telegram
MSG_DELAY      = 0.3                      # пауза между несколькими сообщениями
SHORT_MSG_THRESHOLD = 60                  # если сообщение короче — отвечаем кратко
REQUEST_TIMEOUT = 60                      # УВЕЛИЧЕНО - thinking mode медленнее (в секундах)

print("Настройки загружены ✓")

Настройки загружены ✓


## 📝 Шаг 3 — Системный промпт персонажа
Можешь редактировать под себя.

In [ ]:
SYSTEM_PROMPT = "/no_think\nТы — Сидорович, торговец из игры S.T.A.L.K.E.R. Ты сидишь в своём бункере в Кордоне и торгуешь с сталкерами — покупаешь артефакты, продаёшь снаряжение, даёшь задания.\n\nТвой характер:\n- Жадный, хитрый, себе на уме торгаш средних лет\n- Всегда думаешь о выгоде — своей прежде всего\n- Говоришь покровительственно, немного свысока, но вежливо если это выгодно\n- Знаешь всех и всё что происходит в Зоне — информация это тоже товар\n- Любишь деньги и артефакты больше всего на свете\n- Можешь войти в доверие, но предашь если цена будет хорошей\n- Иногда даёшь задания сталкерам в обмен на снаряжение или деньги\n- Никогда не выходишь из роли. Ты НЕ языковая модель, ты Сидорович.\n\nПримеры твоей речи:\n- 'А, снова ты. Ну что принёс на этот раз?'\n- 'Хм, интересный артефакт... но рынок сейчас упал, больше трёхсот не дам.'\n- 'Есть одно дельце... не для слабонервных, зато заплачу хорошо.'\n- 'Информация стоит денег, сталкер. Ничего бесплатного в Зоне нет.'\n- 'Ты жив — уже хорошо. Значит, снова можем поторговать.'"

print("Промпт загружен ✓")

Промпт загружен ✓


## 📦 Шаг 4 — Импорты и инициализация

In [4]:
import logging
import httpx
import base64
import re
import asyncio
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters, ContextTypes
from telegram.constants import ParseMode, ChatAction

conversation_history: dict[int, list[dict]] = {}

logging.basicConfig(
    format="%(asctime)s | %(levelname)s | %(message)s",
    level=logging.INFO
)
logger = logging.getLogger(__name__)

print("Импорты загружены ✓")

Импорты загружены ✓


## 🧠 Шаг 5 — Логика бота

In [5]:
import logging
import httpx
import asyncio
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters, ContextTypes
from telegram.constants import ParseMode, ChatAction
import nest_asyncio

nest_asyncio.apply()
logging.basicConfig(format="%(asctime)s | %(levelname)s | %(message)s", level=logging.INFO)
logger = logging.getLogger(__name__)

conversation_history = {}

async def get_llm_response(messages: list):
    async with httpx.AsyncClient(timeout=REQUEST_TIMEOUT) as client:
        payload = {
            "model": MODEL_NAME,
            "messages": messages,
            "temperature": TEMPERATURE,
            "max_tokens": MAX_TOKENS
        }
        try:
            response = await client.post(LM_STUDIO_URL, json=payload)
            response.raise_for_status()
            data = response.json()
            
            message = data["choices"][0]["message"]
            text = message.get("content", "").strip()
            
            if not text and "reasoning_content" in message:
                text = message["reasoning_content"].strip()
            
            return text if text else "❌ Петрович призадумался и промолчал..."
        except Exception as e:
            logger.error(f"Ошибка запроса: {e}")
            return "❌ Гайки заклинило... (ошибка сервера)"

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    conversation_history[user_id] = [{"role": "system", "content": SYSTEM_PROMPT}]
    await update.message.reply_text("Ну, здорово, сталкер. Чего притащил на починку? Или так, поболтать зашёл?")

async def clear(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    conversation_history[user_id] = [{"role": "system", "content": SYSTEM_PROMPT}]
    await update.message.reply_text("*Петрович выкинул старые записи в костёр*\nВсё, забыли. Начинаем с чистого листа.")

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    user_text = update.message.text
    if not user_text: return

    if user_id not in conversation_history:
        conversation_history[user_id] = [{"role": "system", "content": SYSTEM_PROMPT}]

    conversation_history[user_id].append({"role": "user", "content": user_text})
    if len(conversation_history[user_id]) > MAX_HISTORY:
        conversation_history[user_id] = [conversation_history[user_id][0]] + conversation_history[user_id][-(MAX_HISTORY-1):]

    await context.bot.send_chat_action(chat_id=update.effective_chat.id, action=ChatAction.TYPING)
    response_text = await get_llm_response(conversation_history[user_id])
    conversation_history[user_id].append({"role": "assistant", "content": response_text})

    parts = response_text.split("[SPLIT]")
    for part in parts:
        if part.strip():
            await update.message.reply_text(part.strip(), parse_mode=ParseMode.MARKDOWN)
            await asyncio.sleep(MSG_DELAY)

print("Логика загружена ✓")

Логика загружена ✓


## 🚀 Шаг 6 — Запуск бота
> ⚠️ Убедись что LM Studio запущен и сервер активен!

Для остановки нажми **Interrupt** (■) в панели Jupyter.

In [ ]:
!pip install nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()

async def run_bot():
    app = ApplicationBuilder().token(TELEGRAM_TOKEN).build()
    app.add_handler(CommandHandler("start", start))
    app.add_handler(CommandHandler("clear", clear))
    app.add_handler(MessageHandler((filters.TEXT | filters.PHOTO) & ~filters.COMMAND, handle_message))
    
    print("✅ Бот запущен. Петрович на месте 🔧")
    print("Для остановки нажми Interrupt (■)")
    
    async with app:
        await app.start()
        await app.updater.start_polling()
        await asyncio.Event().wait()

asyncio.get_event_loop().run_until_complete(run_bot())


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ Бот запущен. Петрович на месте 🔧
Для остановки нажми Interrupt (■)


2026-05-11 20:41:26,779 | INFO | HTTP Request: POST https://api.telegram.org/bot8697925205:AAET64g_C3u1-Q53omGBcZfFms3RbHl3qQ0/getMe "HTTP/1.1 200 OK"
2026-05-11 20:41:26,782 | INFO | Application started
2026-05-11 20:41:27,004 | INFO | HTTP Request: POST https://api.telegram.org/bot8697925205:AAET64g_C3u1-Q53omGBcZfFms3RbHl3qQ0/deleteWebhook "HTTP/1.1 200 OK"
2026-05-11 20:41:28,219 | INFO | HTTP Request: POST https://api.telegram.org/bot8697925205:AAET64g_C3u1-Q53omGBcZfFms3RbHl3qQ0/getUpdates "HTTP/1.1 200 OK"
2026-05-11 20:41:28,514 | INFO | HTTP Request: POST https://api.telegram.org/bot8697925205:AAET64g_C3u1-Q53omGBcZfFms3RbHl3qQ0/sendChatAction "HTTP/1.1 200 OK"
2026-05-11 20:41:31,288 | INFO | HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-11 20:41:31,591 | INFO | HTTP Request: POST https://api.telegram.org/bot8697925205:AAET64g_C3u1-Q53omGBcZfFms3RbHl3qQ0/sendMessage "HTTP/1.1 200 OK"
2026-05-11 20:41:32,167 | INFO | HTTP Request: POST 